# Phase 1: Courier Problem Design and Planning

**Student:** Tanishq  
Mini Courier Planner — design of data, graph format, and algorithm choices.

## Problem statement

A courier company receives parcels that must be delivered to customer locations. Each parcel has a destination, weight, deadline, priority, and value. One vehicle has limited carrying capacity, so the system must choose which parcels to load for one trip and then order those destinations into a delivery route that returns to the warehouse.

**Objective:** design a prototype that can prioritize parcels, select an optimal packing plan, compute shortest road paths, and produce an approximate tour.

## Data design

| Attribute | Meaning |
|---|---|
| `parcel_id` | Unique parcel identifier |
| `destination` | Delivery location code |
| `weight` | Capacity cost in kg |
| `value` | Business value used in the benefit score |
| `deadline` | Required day; smaller is more urgent |
| `priority` | High, Medium, or Low |
| `benefit` | `value + 20×priority_score + 5×max(0, 6−deadline)` |

Graph CSV format: `source,destination,distance` with non-negative kilometres. The graph is undirected. `Warehouse` is the depot.

## How the pieces work together

```text
parcels.csv                    routes.csv
    |                              |
    v                              v
Timsort by priority,           undirected adjacency list
then earliest deadline
    |
    v
0/1 knapsack DP (exact packing under capacity)
    |
    v
selected destinations  -------> Dijkstra shortest paths
                                nearest-neighbor TSP tour
                                      |
                                      v
                         packing plan + ordered route + km
```

## Algorithm selection

| Component | Algorithm | Why it fits |
|---|---|---|
| Prioritization | Timsort (`sorted`) | Stable two-key order, O(n log n) |
| Box filling | 0/1 knapsack DP | Exact optimum for integer kg capacity |
| Shortest paths | Dijkstra + heap | Non-negative road distances |
| Route | Nearest-neighbor | Fast TSP approximation; not globally optimal |

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "app.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from app import load_graph, load_parcels
from src.sorting import parcel_benefit

parcels = load_parcels(ROOT / "datasets/parcels.csv")
graph = load_graph(ROOT / "datasets/routes.csv")
print(f"Loaded {len(parcels)} parcels and {len(graph)} graph nodes.")
print("Sample parcels:")
for parcel in parcels[:4]:
    print(f"  {parcel.parcel_id} -> {parcel.destination} | {parcel.priority} | {parcel.weight} kg | benefit {parcel_benefit(parcel)}")
print("Warehouse neighbors:", graph["Warehouse"])

## Phase 1 summary

Phase 1 freezes the courier problem, CSV schemas, benefit rule, and the algorithm pipeline used in later phases.